# Optimization in Machine Learning

<div class="alert alert-warning">     
<b> Table of Contents </b>

1. [Introduction](#introduction)
2. [K Fold](#kfold)
3. [GridsearchCV](#grid)
5. [Exercise](#exercise)

<div class="alert alert-warning">
    <b> I. <i>Introduction </i></b>
    <a id="introduction"></a></div>

In machine learning, optimization refers to the process of improving a model’s performance by finding the best possible configuration of its parameters and hyperparameters. The goal is to build a model that not only fits the training data well but also generalizes effectively to unseen data.

In [67]:
from sklearn.datasets import load_iris
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC as svm

We will use again the iris dataset.

In [68]:
iris = load_iris()
X = iris.data[:,:2]
y = (iris.target == 0).astype(int) 

<div class="alert alert-warning">
    <b> II. <i>K-Folding </i></b>
    <a id="kfold"></a></div>

K-Fold Cross Validation is a model evaluation technique that helps obtain a more robust measure of the performance of a classification model, especially when the available dataset is limited. This technique splits the dataset into k subsets or "folds," and uses each subset multiple times to train and validate the model. This way, a more reliable estimate of the model’s ability to generalize to new data can be obtained.

A model can perform well on training data but poorly on new data (overfitting). To avoid this, we use K-Fold Cross Validation.

To perform this task, we again use scikit-learn — specifically, the KFold function. This function has the following parameters:

- `n_splits`: Number of splits to make.

- `shuffle`: Boolean indicating whether the data should be shuffled before splitting.

- `random_state`: Random seed.

It returns the different train and test splits, ensuring that the training set and test set sizes follow the specified distribution.

We will use the function ```KFold`` to do cross validation. This function takes these parameters: 
- ``n_splits``: how many folds you want.
- ``shuffle``: Before splitting the data, it is shuffled randomly. 
- ``random_state``: To control randomness. 

We can define a function that performs: 
- Kfolding 
- Training and Testing spliting
- Train a model 
- Test it and evaluate performance. 

Returning the average performance, the results of each fold and the best model.

In [69]:
def cross_validate(model, X, y, k=5):
     # 1. We create a K-Fold object that will split the data into 5 parts, 
     # shuffle the dataset before splitting and use fixed seed
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    # Variables to store results
    scores = [] 
    best_score = -1 
    best_model = None

    # Kfold returns the indices of the rows used for training and testing. 
    for train_index, test_index in kf.split(X):
        #split data
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # train model
        model.fit(X_train, y_train)

        # make prediction
        y_pred = model.predict(X_test)

        # evaluate performance
        score = accuracy_score(y_test, y_pred)
        scores.append(score)

        #store result
        if score > best_score:
            best_score = score
            best_model = model

    return np.mean(scores), scores, best_model #<- We return the mean accuracy, all fold accuracy and best model

Let's see how this works. We create a Logistic Regression model.

In [70]:
model = LogisticRegression()

And compute cross validation:

In [71]:
mean_score, scores, best_model = cross_validate(model, X, y, 5)
print(f"Mean score:{mean_score}")
print(f"Scores: {scores}")

Mean score:1.0
Scores: [1.0, 1.0, 1.0, 1.0, 1.0]


What can we do with the best_model variable? 

We can print the hyperparameters with we trained the model. 

In [72]:
print(best_model.get_params())

{'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 100, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}


<div class="alert alert-warning">
    <b> III. <i>Gridsearch </i></b>
    <a id="grid"></a></div>

In the training of machine learning models, a hyperparameter is any parameter that is not directly learned during the training process but must be defined before training begins. These parameters are configurations that influence the behavior of the model, as well as its ability to learn and generalize on new datasets.

Unlike parameters that are determined during the training of the model (such as weights in a neural network), hyperparameters must be set beforehand, typically based on the designer's experience, trial and error, or through search techniques.

GridSearchCV automates the search for the best combination of hyperparameters by:

- Trying all possible combinations from a predefined grid
- Evaluating each using cross-validation
- Selecting the configuration with the best average performance

This ensures the model is not only trained, but also systematically optimized.

In order to use GridSearchCV, we need to create a dictionary of all the parameters we want to try our model:

In [73]:
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

Then, we can create an object ``GridSearchCV`` with the following parameters: 
- ``estimator``:The model you want to optimize.
- ``param_grid``: Dictionary of parameters to try.
- ``cv``: Use 5-fold cross validation.
- ``scoring``: Metric used to compare models. We can indicate accuracy, precision, recall,F1.
- ``verbose``: Prints progress while running if set to 1.

In [74]:
model = svm()
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1
)

Once we have the object created, we call ``fit`` to mtrain the grid search:

In [75]:
grid.fit(X, y)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10], 'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf']},
             scoring='accuracy', verbose=1)

To see the results we can call to ``best_params`` and ``best_score``:

In [76]:
print("Best parameters:", grid.best_params_)
print("Best accuracy:", grid.best_score_)

Best parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Best accuracy: 1.0


We can obtain the best model calling ``best_estimator_``

In [77]:
best_model = grid.best_estimator_

Then, we can perform predictions and compute more metrics: 

In [78]:
predictions = best_model.predict(X)
print("\nReport:\n", classification_report(y, predictions))


Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00        50

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150



<div class="alert alert-warning">
    <b> IV. <i>Exercise </i></b>
    <a id="Exercise"></a></div>